# 03 - Web Scraping: Inside Airbnb

**Why this step:** the data quality check showed weak correlation between price and the existing features
(only `attr_index_norm` exceeded |0.3|). Following the project hint, we enrich the data with extra features
(amenities, number of reviews, ratings, host and property details, availability).

**Why Inside Airbnb and not Airbnb directly:**
- Airbnb's Terms of Service prohibit bots, crawlers and scrapers.
- The Kaggle dataset has no listing IDs or URLs, so its rows cannot be matched to Airbnb pages anyway.
- [Inside Airbnb](https://insideairbnb.com/get-the-data/) publishes Airbnb listing data under a
  **Creative Commons Attribution 4.0** license, so using it is allowed.

**What this notebook does:**
1. Uses **Selenium** to open the Inside Airbnb download page in Chrome and collect the download links.
2. Keeps only the latest `listings.csv.gz` and `neighbourhoods.geojson` for our 10 cities.
3. Downloads the files into `data/external/insideairbnb/` and saves a record of the sources.

*Data source: Inside Airbnb (insideairbnb.com), licensed under CC BY 4.0.*

In [3]:
import re
import time
from pathlib import Path
from urllib.parse import unquote

import pandas as pd
import requests
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

## Step 1: Settings
Each city is paired with its country as it appears in the Inside Airbnb URLs.
Checking the country avoids picking a different city that has the same name.

In [4]:
URL = "https://insideairbnb.com/get-the-data/"

CITY_COUNTRY = {
    "amsterdam": "the-netherlands",
    "athens": "greece",
    "barcelona": "spain",
    "berlin": "germany",
    "budapest": "hungary",
    "lisbon": "portugal",
    "london": "united-kingdom",
    "paris": "france",
    "rome": "italy",
    "vienna": "austria",
}

WANTED_FILES = ["data/listings.csv.gz", "visualisations/neighbourhoods.geojson"]

OUT_DIR = Path("../data/external/insideairbnb")
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Step 2: Open the page with Selenium and collect all download links
- `--headless=new` runs Chrome without opening a visible window.
- `WebDriverWait` waits until the links have loaded instead of reading the page too early.
- Selenium downloads the matching ChromeDriver automatically (Selenium Manager), so only Chrome needs to be installed.

In [5]:
options = Options()
options.add_argument("--headless=new")
options.add_argument("--window-size=1400,1000")

driver = webdriver.Chrome(options=options)
try:
    driver.get(URL)
    WebDriverWait(driver, 60).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "a[href*='data.insideairbnb.com']"))
    )
    links = driver.find_elements(By.CSS_SELECTOR, "a[href*='data.insideairbnb.com']")
    hrefs = [a.get_attribute("href") for a in links]
finally:
    driver.quit()  # always close the browser, even if an error happened

print("Download links found on the page:", len(hrefs))

Download links found on the page: 861


## Step 3: Keep only our 10 cities and the latest snapshot of each file
Every link has this shape:
`https://data.insideairbnb.com/<country>/<region>/<city>/<date>/<file>`
The regular expression below extracts the city, the date and the file name from each link.

In [6]:
PATTERN = re.compile(r"/(?P<city>[^/]+)/(?P<date>\d{4}-\d{2}-\d{2})/(?P<file>.+)$")

latest = {}  # (city, file) -> (date, url)
for href in hrefs:
    readable = unquote(href)  # turns %C3%B6 back into letters such as ö
    m = PATTERN.search(readable)
    if not m:
        continue
    city, date, file = m["city"], m["date"], m["file"]
    if city not in CITY_COUNTRY or f"/{CITY_COUNTRY[city]}/" not in readable:
        continue
    if file not in WANTED_FILES:
        continue
    key = (city, file)
    if key not in latest or date > latest[key][0]:
        latest[key] = (date, href)

sources = pd.DataFrame(
    [{"city": c, "file": f, "snapshot_date": d, "url": u} for (c, f), (d, u) in latest.items()]
).sort_values(["city", "file"]).reset_index(drop=True)

missing = sorted(set(CITY_COUNTRY) - set(sources["city"]))
print("Cities found:", sources["city"].nunique(), "/ 10")
print("Missing cities:", missing if missing else "none")
sources

Cities found: 10 / 10
Missing cities: none


,city,file,snapshot_date,url
0,amsterdam,data/listings.csv.gz,2026-09-16,https://data.insideairbnb.com/the-netherlands/...
1,amsterdam,visualisations/neighbourhoods.geojson,2026-09-16,https://data.insideairbnb.com/the-netherlands/...
2,athens,data/listings.csv.gz,2026-09-22,https://data.insideairbnb.com/greece/attica/at...
3,athens,visualisations/neighbourhoods.geojson,2026-09-22,https://data.insideairbnb.com/greece/attica/at...
4,barcelona,data/listings.csv.gz,2026-09-17,https://data.insideairbnb.com/spain/catalonia/...
5,barcelona,visualisations/neighbourhoods.geojson,2026-09-17,https://data.insideairbnb.com/spain/catalonia/...
6,berlin,data/listings.csv.gz,2026-09-19,https://data.insideairbnb.com/germany/be/berli...
7,berlin,visualisations/neighbourhoods.geojson,2026-09-19,https://data.insideairbnb.com/germany/be/berli...
8,budapest,data/listings.csv.gz,2026-09-21,https://data.insideairbnb.com/hungary/k%C3%B6z...
9,budapest,visualisations/neighbourhoods.geojson,2026-09-21,https://data.insideairbnb.com/hungary/k%C3%B6z...


## Step 4: Download the files
- Files are downloaded in chunks (`stream=True`) so large files do not fill the memory.
- Files that already exist are skipped, so the cell can be re-run safely.
- A short pause between downloads keeps the load on the server polite.

In [7]:
def download(url, dest):
    if dest.exists():
        print(f"skip (already downloaded): {dest.name}")
        return
    with requests.get(url, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                f.write(chunk)
    print(f"downloaded: {dest.name}  ({dest.stat().st_size / 1e6:.1f} MB)")


local_paths = []
for row in sources.itertuples():
    dest = OUT_DIR / f"{row.city}_{row.snapshot_date}_{Path(row.file).name}"
    download(row.url, dest)
    local_paths.append(str(dest))
    time.sleep(1)

sources["local_path"] = local_paths
sources.to_csv(OUT_DIR / "sources.csv", index=False)
print("\nSaved the list of sources to", OUT_DIR / "sources.csv")

downloaded: amsterdam_2026-09-16_listings.csv.gz  (5.9 MB)
downloaded: amsterdam_2026-09-16_neighbourhoods.geojson  (0.2 MB)
downloaded: athens_2026-09-22_listings.csv.gz  (8.4 MB)
downloaded: athens_2026-09-22_neighbourhoods.geojson  (0.1 MB)
downloaded: barcelona_2026-09-17_listings.csv.gz  (8.2 MB)
downloaded: barcelona_2026-09-17_neighbourhoods.geojson  (0.4 MB)
downloaded: berlin_2026-09-19_listings.csv.gz  (7.0 MB)
downloaded: berlin_2026-09-19_neighbourhoods.geojson  (0.8 MB)
downloaded: budapest_2026-09-21_listings.csv.gz  (6.3 MB)
downloaded: budapest_2026-09-21_neighbourhoods.geojson  (0.2 MB)
downloaded: lisbon_2026-09-19_listings.csv.gz  (14.1 MB)
downloaded: lisbon_2026-09-19_neighbourhoods.geojson  (4.3 MB)
downloaded: london_2026-09-17_listings.csv.gz  (48.3 MB)
downloaded: london_2026-09-17_neighbourhoods.geojson  (1.0 MB)
downloaded: paris_2026-09-16_listings.csv.gz  (40.3 MB)
downloaded: paris_2026-09-16_neighbourhoods.geojson  (0.1 MB)
downloaded: rome_2026-09-17_lis

## Step 5: Quick check of one file
Confirm the new features we need are actually in the data.

In [8]:
sample_path = sources.loc[sources["file"] == "data/listings.csv.gz", "local_path"].iloc[0]
sample = pd.read_csv(sample_path, low_memory=False)  # pandas reads .gz files directly

print("File:", Path(sample_path).name)
print("Shape:", sample.shape)

needed = [
    "neighbourhood_cleansed", "latitude", "longitude", "property_type", "room_type",
    "amenities", "number_of_reviews", "review_scores_rating", "host_is_superhost",
    "host_listings_count", "availability_365", "price",
]
print("\nNeeded columns present:")
for col in needed:
    print(f"  {col:<25} {'yes' if col in sample.columns else 'MISSING'}")

File: amsterdam_2026-09-16_listings.csv.gz
Shape: (10612, 90)

Needed columns present:
  neighbourhood_cleansed    yes
  latitude                  yes
  longitude                 yes
  property_type             yes
  room_type                 yes
  amenities                 yes
  number_of_reviews         yes
  review_scores_rating      yes
  host_is_superhost         yes
  host_listings_count       yes
  availability_365          yes
  price                     yes
